# SBS Radar · promoción de un refresco al runtime

Esta ejecución usa los seis PDF sellados de ambas familias y los vectores reales existentes. Prepara SK03/SK04/SK06, carga el nuevo catálogo y comprueba una promoción fallida conservando el anterior. No invoca endpoints ni acredita operación cloud, UI o interpretación jurídica. Las comparaciones por página son parciales; no equivalen a cambios materiales por artículo.

In [ ]:
from pathlib import Path
from dataclasses import replace
from tempfile import TemporaryDirectory
import hashlib, json, sys
ROOT=Path.cwd().resolve()
assert (ROOT/'src/sbs/runtime.py').is_file(), 'Ejecuta el notebook desde la raíz del proyecto SBS Radar.'
sys.path.insert(0,str(ROOT/'src'))
from sbs.operations import RefreshRunner, load_sealed_plan
from sbs.operations.preparers import build_real_hooks
from sbs.runtime import LocalService
config=json.loads((ROOT/'config/genie-pilot-002.json').read_text())
pairs={c['pair']['pair_id']:c['pair'] for c in config['contexts']}
plan=replace(load_sealed_plan(ROOT),pairs=tuple(pairs.values()))
protected=list(plan.originals.values())+[ROOT/'data/retrieval/sk04-real-001/index.json',ROOT/'data/retrieval/sk04-real-001/records.json']
hashes=lambda: {str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
before=hashes()
workspace=TemporaryDirectory(prefix='sbs-runtime-notebook-')
# La ruta temporal creada por el servidor se canoniza antes del lector no-follow.
state=Path(workspace.name).resolve()/'state'
runner=RefreshRunner(plan,state)
prepared=runner.run(run_id='runtime-notebook',hooks=build_real_hooks(ROOT),force_revalidate=True)
assert prepared['status']=='published'
pointer=runner.current()
print(json.dumps({'preparation':prepared['status'],'sources':len(plan.originals),'cloud':False,'new_embedding_calls':0}))

In [ ]:
service=LocalService()
previous_snapshot=service.snapshot
old_catalog=service.catalog()
service.promote_release(state,pointer=pointer)
new_catalog=service.catalog()
assert {p['family_id'] for p in new_catalog['pairs']}=={'cybersecurity','market_conduct'}
assert service.snapshot!=previous_snapshot
assert new_catalog!=old_catalog
checked=[]
for pair in new_catalog['pairs']:
    selected=pair['provisions'][0]['id']
    comparison=service.comparison(pair['id'],selected)
    assert comparison['evidence_status']=='partial'
    assert comparison['before']['text'] or comparison['after']['text']
    for side in ('before','after'):
        if comparison[side].get('source_id'):
            assert service.source(comparison[side]['source_id']).read_bytes().startswith(b'%PDF')
    checked.append({'family':pair['family_id'],'pair':pair['id'],'selection':selected,'provisions':len(pair['provisions'])})
print(json.dumps({'runtime_promoted':True,'snapshot':service.snapshot,'comparisons':checked},ensure_ascii=False))

In [ ]:
# HTTP real sobre la app con el runtime promovido; no sustituye prueba visual UI.
import socket, threading, time
import requests, uvicorn
from sbs.webapp import create_app
listener=socket.socket()
listener.bind(('127.0.0.1',0))
port=listener.getsockname()[1]
server=uvicorn.Server(uvicorn.Config(create_app(service),log_level='critical',lifespan='off'))
thread=threading.Thread(target=server.run,kwargs={'sockets':[listener]},daemon=True)
thread.start()
http_evidence=[]
try:
    deadline=time.monotonic()+5
    while not server.started and thread.is_alive() and time.monotonic()<deadline:
        time.sleep(0.05)
    assert server.started, 'La app local no inició.'
    base='http://127.0.0.1:'+str(port)
    with requests.Session() as client:
        client.trust_env=False
        catalog=client.get(base+'/api/catalog',timeout=5)
        assert catalog.status_code==200 and catalog.json()['pairs']==new_catalog['pairs']
        for pair in new_catalog['pairs']:
            selected=pair['provisions'][0]['id']
            response=client.get(base+'/api/comparison',params={'pair_id':pair['id'],'provision_id':selected},timeout=5)
            assert response.status_code==200
            body=response.json()
            assert body['before']['text']==service.comparison(pair['id'],selected)['before']['text']
            source_id=body['before']['source_id']
            source=client.get(base+'/api/sources/'+source_id,timeout=5)
            assert source.status_code==200 and hashlib.sha256(source.content).hexdigest()==hashlib.sha256(service.source(source_id).read_bytes()).hexdigest()
            http_evidence.append({'family':pair['family_id'],'comparison_http':response.status_code,'source_http':source.status_code,'source_sha256':hashlib.sha256(source.content).hexdigest()})
    print(json.dumps({'runtime_http':http_evidence,'ui_visual_verified':False,'generation_called':False},ensure_ascii=False))
finally:
    server.should_exit=True
    thread.join(timeout=5)
    listener.close()
    assert not thread.is_alive(), 'Servidor de prueba aún activo.'


In [ ]:
import copy
published_snapshot=service.snapshot
published_catalog=service.catalog()
bad=copy.deepcopy(pointer)
bad['sha256']='0'*64
try:
    service.promote_release(state,pointer=bad)
except ValueError:
    rejected=True
else:
    rejected=False
assert rejected and service.snapshot==published_snapshot and service.catalog()==published_catalog
assert before==hashes()
print(json.dumps({'invalid_release_rejected':True,'previous_runtime_preserved':True,'protected_files_unchanged':len(before),'generation_calls':0,'cloud_e2e_validated':False}))
workspace.cleanup()